<a href="https://colab.research.google.com/github/allnameswhy/arctic_keywords/blob/main/untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

테스트 구문

In [ ]:
import re
test = "“Stable isotope  record and depositional environments in the  Antarctic Polar Front of Drake Passage,  Western Antarctica”"
year_re = re.compile("\d{4}\.")
title_re = re.compile("“[\w\W]*”")
title_re.search(test)

<re.Match object; span=(0, 122), match='“Stable isotope  record and depositional environm>

1) 먼저 아래 코드를 실행하여 분석하고자 하는 텍스트를 업로드하세요. <br>
파일 제목을 vol0[숫자]으로 하여야 합니다.

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving vol11.txt to vol11.txt


2) 다음 아래 코드를 실행하여 NLTK를 설치합니다. <br>
첫 프롬프트에 "d", 두번째 프롬프트에 "all"로 답합니다.

In [ ]:
import re
import nltk
nltk.download()


NLTK Downloader
---------------------------------------------------------------------------
    d) Download   l) List    u) Update   c) Config   h) Help   q) Quit
---------------------------------------------------------------------------
Downloader> d

Download which package (l=list; x=cancel)?
  Identifier> all


       | 
       | Downloading package abc to /root/nltk_data...
       |   Unzipping corpora/abc.zip.
       | Downloading package alpino to /root/nltk_data...
       |   Unzipping corpora/alpino.zip.
       | Downloading package averaged_perceptron_tagger to
       |     /root/nltk_data...
       |   Unzipping taggers/averaged_perceptron_tagger.zip.
       | Downloading package averaged_perceptron_tagger_ru to
       |     /root/nltk_data...
       |   Unzipping taggers/averaged_perceptron_tagger_ru.zip.
       | Downloading package basque_grammars to /root/nltk_data...
       |   Unzipping grammars/basque_grammars.zip.
       | Downloading package biocreative_ppi to /root/nltk_data...
       |   Unzipping corpora/biocreative_ppi.zip.
       | Downloading package bllip_wsj_no_aux to /root/nltk_data...
       |   Unzipping models/bllip_wsj_no_aux.zip.
       | Downloading package book_grammars to /root/nltk_data...
       |   Unzipping grammars/book_grammars.zip.
       | Downloading 


---------------------------------------------------------------------------
    d) Download   l) List    u) Update   c) Config   h) Help   q) Quit
---------------------------------------------------------------------------
Downloader> q


True

3) 아래 코드를 실행하면 각 단어별 빈도를 출력합니다. <br>
단, 1)에서 업로드한 파일 제목과 아래 raw = open("vol0[숫자].txt" ... ) 가 일치하는지 확인하고, <br>
볼륨 넘버를 vol = "0[숫자]"으로 지정해줍니다.

In [ ]:
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag
from nltk import Text
from nltk import FreqDist
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import csv

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return None

# STDIN으로 vol받아서 해당되는 텍스트파일 열기
vol = input()
vol_str = "vol{}.txt".format(vol)
raw = open(vol_str, "r", encoding="CP949")
library = []

# 각 줄에서 attribute 힌트를 주는 문자열 목록
year_re = re.compile("(^\d{4}-\d{4}$)|(^\d{4}$)")
num_re = re.compile("^\d-\d+")
jour_re = re.compile(":\s+\d+-\d+\.")
page_re = re.compile("^\d+$")
doi_re = re.compile("^doi:")
part_re = re.compile("^[Pp][Aa][Rr][Tt] \d")
abs_re = re.compile("ABSTRACTS")
year_in_num_re = re.compile("\d{4}")
title_in_num_re = re.compile("[“\"][\w\W]*[”\"]")
vol_re = re.compile("(11)|(12)")


# 어디를 읽고있나요?
is_title = False
is_abs = False
abs_temp = ''
title_temp = ''

for line in raw:
  line = line.strip()

  if part_re.search(line): # 다음 라인을 읽어 category값 넘김
    next_line = next(raw).strip()
    while not next_line:
      next_line = next(raw).strip()
    category = next_line
    continue

  elif num_re.search(line): # 여기서부터 새로운 엔트리니까 지금까지의 데이터를 저장
    if not is_title and is_abs:
      a = {}
      a['vol'] = vol
      a['year'] = year
      a['category'] = category
      a['title'] = title
      a['authors'] = authors
      a['journal'] = journal
      a['abstract'] = abs_temp.strip()
      library.append(a)

    is_abs = False
    is_title = True
    abs_temp = ''

  elif not line or doi_re.search(line) or year_re.search(line) or page_re.search(line) or abs_re.search(line): # 무시해야 할 라인
    continue

  if is_title and not is_abs: # 지금 타이틀을 읽고 있음
    if not title_temp:  # 현재 줄을 title_temp에 저장
      title_temp = line
    else:
      if vol_re.search(vol):
        title_temp = '!'.join([title_temp, line])
      else:
        title_temp = ' '.join([title_temp, line])

    if jour_re.search(line): # 여기가 타이틀의 끝이므로 title_temp를 처리하고 초기화
      title_temp = re.sub(num_re,'', title_temp).strip()
      #print(title_temp)
      if vol_re.search(vol):
        meta_list = title_temp.split("!")
        title = meta_list[1].strip()
        authors = meta_list[2].strip()
        pointer1 = year_in_num_re.search(meta_list[3]).span()
        year = meta_list[3][pointer1[0] : pointer1[1]]
        journal = meta_list[3][0 : pointer1[0] - 1] + meta_list[3][pointer1[1] + 1 : len(meta_list[3]) - 1]
      else:
        meta_list = []
        pointer1 = year_in_num_re.search(title_temp).span()
        pointer2 = title_in_num_re.search(title_temp).span()
        meta_list.append(title_temp[0 : pointer1[0] - 1])
        meta_list.append(title_temp[pointer1[0] : pointer1[1]])
        meta_list.append(title_temp[pointer2[0] + 1 : pointer2[1] - 1])
        meta_list.append(title_temp[pointer2[1] + 1 : len(title_temp) - 1])
        authors = meta_list[0].strip()
        year = meta_list[1].strip()
        title = meta_list[2].strip()
        journal = meta_list[3].strip()

      is_title = False
      is_abs = True
      title_temp = ''

  elif not is_title and is_abs: # 지금 본문을 읽고 있음
    if abs_temp: # 현재 줄을 abs_temp에 저장
      abs_temp = ' '.join([abs_temp, line])
    else:
      abs_temp = line

raw.close()

if not is_title and is_abs: # 가장 마지막 엔트리
  a = {}
  a['vol'] = vol
  a['year'] = year
  a['category'] = category
  a['title'] = title
  a['authors'] = authors
  a['journal'] = journal
  a['abstract'] = abs_temp.strip()
  library.append(a)

#print(library)

# library에 있는 abs만 모아서 코퍼스를 형성
corpus = ''
corpuses = {}
cur = ''
prev = ''
#for item in library:
#  if item['abstract']:
#    corpus = corpus + item['abstract']
for i in range(len(library)):
  cur = library[i]['category']
  if i == 0 or cur == prev:
    corpus = corpus + library[i]['abstract'] + ' '
  else:
    corpuses[prev] = corpus
    corpus = library[i]['abstract']
  prev = cur
corpuses[prev] = corpus
print(corpuses.keys())

# 문장으로 토큰화
for category in corpuses.keys():
  print(category)
  corpus = corpuses.get(category)
  sentences = sent_tokenize(corpus)
  lm = WordNetLemmatizer()
  sw = set(stopwords.words('english'))
  sw.add("keywords")
  vocab = {}
  proc_sentences = []

  for sentence in sentences: # 하나의 문장 속에서
    words = word_tokenize(sentence)
    words = [w.lower() for w in words]
    words = pos_tag(words)
    proc_sentence = []

    for word in words: # 각 단어를 추출해내 정규화
      #word = word.lower()
      word = lm.lemmatize(word[0], get_wordnet_pos(word[1])) if get_wordnet_pos(word[1]) else lm.lemmatize(word[0])
      if word not in sw and len(word) > 2:
        proc_sentence.append(word)
        if word not in vocab: # vocab에 빈도 기록
          vocab[word] = 0
        vocab[word] += 1

    proc_sentences.append(proc_sentence)

  vocab_sorted = sorted(vocab.items(), key = lambda x: x[1], reverse = True)

  #print(proc_sentences)
  print(vocab_sorted) # 완성된 데이터
  result_str = "{category} {vol}.csv".format(category=category, vol=vol)
  result = open(result_str, 'w', newline="")
  writer = csv.writer(result)
  writer.writerow(["Word", "Frequency"])

  for word in vocab_sorted:
    word_list = [word[0], word[1]]
    writer.writerow(word_list)

  result.close()

  #text = Text(word_tokenize(corpus))
  #top_keys = [x[0] for x in vocab]
  #fd = FreqDist(vocab)
  #text.dispersion_plot(top_keys[0 : 10])
  #wc = WordCloud(width=2000, height=1500, background_color="white", random_state=0)
  #wc_str = "wc{}.png".format(vol)
  #plt.imshow(wc.generate_from_frequencies(fd))
  #plt.axis("off")
  #plt.savefig(wc_str)
  #plt.show() # 워드클라우드


12
dict_keys(['Climate Sciences', 'Geosciences', 'Marine Sciences', 'Life Sciences', 'Policy'])
Climate Sciences
[('ice', 70), ('arctic', 55), ('sea', 54), ('data', 30), ('aerosol', 29), ('particle', 27), ('water', 25), ('study', 24), ('model', 23), ('use', 21), ('ocean', 21), ('sar', 21), ('result', 20), ('analysis', 19), ('summer', 18), ('show', 18), ('cluster', 18), ('low', 17), ('also', 17), ('change', 17), ('region', 16), ('temperature', 16), ('polar', 15), ('effect', 15), ('high', 15), ('rate', 15), ('increase', 15), ('concentration', 15), ('image', 15), ('event', 15), ('distribution', 14), ('atmospheric', 14), ('height', 14), ('subglacial', 13), ('two', 13), ('observation', 13), ('size', 13), ('solution', 13), ('number', 13), ('marine', 13), ('station', 13), ('cloud', 13), ('wave', 13), ('survey', 12), ('system', 12), ('resolution', 12), ('surface', 12), ('condition', 12), ('new', 12), ('npf', 12), ('texture', 12), ('lake', 11), ('formation', 11), ('produce', 11), ('frozen', 11)

# todo
vol 08-14 벡터화 완성 <br>
text object의 context 메소드 이용해서 2-word 3-word 단위의 빈도 측정 <br>
각 abstract 태깅해서 연도별 plot하기?? <br>
사실 위의 plot은... lemmatize된 것과 안된거 비교하고 있다 ㅠㅠ <br>
